# 10 — Recovery Validation

This notebook comes after:

`09_Epsilon_Margin_POSet_Robustness.ipynb`

Purpose:

Check whether structurally stronger countries/profiles are associated with faster observed GDP recovery.

This notebook does:

- read the main profile-level POSet outputs from notebook 07;
- read epsilon tolerance outputs from notebook 08;
- read epsilon-margin robustness outputs from notebook 09;
- compare recovery outcomes across:
  - Pareto vs non-Pareto countries;
  - POSet layers;
  - dominance counts;
  - profile groups;
  - epsilon-frontier vs non-frontier countries;
  - epsilon-margin frontier vs non-frontier countries;
- create validation diagnostics and figures;
- export tables for report interpretation.

This notebook does **not**:

- use recovery as an ordering variable;
- claim causality;
- select final variables;
- create a final scalar Economic Resilience Index.

Important:

Lower recovery value means faster recovery.

Special values from the recovery notebook:

```text
Recovery = 0  → no negative growth in the shock window
Recovery = 20 → not recovered by available data
```

Because those special cases can affect interpretation, this notebook computes multiple validation variants:

- `all_recovery_available`
- `affected_only_exclude_zero`
- `recovered_only_exclude_20`
- `affected_recovered_only_exclude_0_and_20`

In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import shutil
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 350)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

PROFILE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
EPSILON_TOLERANCE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Sensitivity_Country_Level"
EPSILON_MARGIN_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Margin_POSet_Robustness"
PRE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"

PROFILE_COUNTRY_MAP_FILE = PROFILE_POSET_DIR / "profile_country_map.csv"
PROFILE_RUN_SUMMARY_FILE = PROFILE_POSET_DIR / "profile_run_summary.csv"
PROFILE_LAYER_SUMMARY_FILE = PROFILE_POSET_DIR / "profile_layer_summary.csv"
PROFILE_PARETO_COUNTRIES_FILE = PROFILE_POSET_DIR / "profile_pareto_countries.csv"
WORKING_MAIN_PROFILE_COUNTRY_MAP_FILE = PROFILE_POSET_DIR / "working_main_country_profile_map_review.csv"
WORKING_MAIN_PROFILE_RUN_SUMMARY_FILE = PROFILE_POSET_DIR / "working_main_profile_run_summary.csv"
WORKING_MAIN_PARETO_COUNTRIES_FILE = PROFILE_POSET_DIR / "working_main_pareto_countries_review.csv"

EPSILON_COUNTRY_SUMMARY_FILE = EPSILON_TOLERANCE_DIR / "epsilon_country_summary.csv"
EPSILON_RUN_SUMMARY_FILE = EPSILON_TOLERANCE_DIR / "epsilon_run_summary.csv"
WORKING_MAIN_EPSILON_COUNTRY_SUMMARY_FILE = EPSILON_TOLERANCE_DIR / "working_main_epsilon_country_summary.csv"
WORKING_MAIN_EPSILON_RUN_SUMMARY_FILE = EPSILON_TOLERANCE_DIR / "working_main_epsilon_run_summary.csv"

EPSILON_MARGIN_COUNTRY_SUMMARY_FILE = EPSILON_MARGIN_DIR / "epsilon_margin_country_summary.csv"
EPSILON_MARGIN_RUN_SUMMARY_FILE = EPSILON_MARGIN_DIR / "epsilon_margin_run_summary.csv"
WORKING_MAIN_EPSILON_MARGIN_COUNTRY_SUMMARY_FILE = EPSILON_MARGIN_DIR / "working_main_epsilon_margin_country_summary.csv"
WORKING_MAIN_EPSILON_MARGIN_RUN_SUMMARY_FILE = EPSILON_MARGIN_DIR / "working_main_epsilon_margin_run_summary.csv"

PRE_POSET_SNAPSHOT_FILE = PRE_POSET_DIR / "pre_poset_analysis_ready_snapshot_full.csv"
CANDIDATE_VARIABLE_SETS_FILE = PRE_POSET_DIR / "candidate_variable_sets.csv"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Recovery_Validation"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "10_Recovery_Validation"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Recovery_Validation"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Profile POSet folder:", PROFILE_POSET_DIR.resolve())
print("Epsilon tolerance folder:", EPSILON_TOLERANCE_DIR.resolve())
print("Epsilon margin folder:", EPSILON_MARGIN_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())
print("Figures folder:", FIGURES_DIR.resolve())

Run ID: 20260622_082628
Profile POSet folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Profile_POSet_Main
Epsilon tolerance folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Epsilon_Sensitivity_Country_Level
Epsilon margin folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Epsilon_Margin_POSet_Robustness
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Recovery_Validation
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\10_Recovery_Validation
Figures folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q

In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

MAIN_SET_NAME = "baseline_6_variables"
WORKING_MAIN_PROFILE_LEVEL = 5

# Use these epsilon values for compact working-main validation summaries.
# Keep all epsilon values in raw outputs, but highlight these.
EPSILON_VALUES_FOR_REVIEW = [0.00, 0.05, 0.10]
EPSILON_MARGIN_VALUES_FOR_REVIEW = [0.00, 0.05, 0.10]

VALIDATION_VARIANTS = [
    {
        "validation_variant": "all_recovery_available",
        "description": "All countries with non-missing recovery.",
        "min_recovery": None,
        "max_recovery": None,
        "exclude_zero": False,
        "exclude_twenty": False,
    },
    {
        "validation_variant": "affected_only_exclude_zero",
        "description": "Exclude Recovery=0 no-negative-growth special cases.",
        "min_recovery": None,
        "max_recovery": None,
        "exclude_zero": True,
        "exclude_twenty": False,
    },
    {
        "validation_variant": "recovered_only_exclude_20",
        "description": "Exclude Recovery=20 not-recovered special cases.",
        "min_recovery": None,
        "max_recovery": None,
        "exclude_zero": False,
        "exclude_twenty": True,
    },
    {
        "validation_variant": "affected_recovered_only_exclude_0_and_20",
        "description": "Exclude both no-shock zeros and not-recovered 20s.",
        "min_recovery": None,
        "max_recovery": None,
        "exclude_zero": True,
        "exclude_twenty": True,
    },
]

print("Main set:", MAIN_SET_NAME)
print("Working profile level:", WORKING_MAIN_PROFILE_LEVEL)
print("Validation variants:")
display(pd.DataFrame(VALIDATION_VARIANTS))

Main set: baseline_6_variables
Working profile level: 5
Validation variants:


,validation_variant,description,min_recovery,max_recovery,exclude_zero,exclude_twenty
0,all_recovery_available,All countries with non-missing recovery.,None,None,False,False
1,affected_only_exclude_zero,Exclude Recovery=0 no-negative-growth special ...,None,None,True,False
2,recovered_only_exclude_20,Exclude Recovery=20 not-recovered special cases.,None,None,False,True
3,affected_recovered_only_exclude_0_and_20,Exclude both no-shock zeros and not-recovered ...,None,None,True,True


In [3]:
# ------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------

def require_file(path, label, required=True):
    path = Path(path)
    if required and not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def read_csv_if_exists(path, label, required=False):
    path = Path(path)

    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded {label}: {len(df)} rows")
        return df

    if required:
        raise FileNotFoundError(f"{label} not found: {path}")

    print(f"Optional file not found: {label} -> {path}")
    return pd.DataFrame()


def clean_country_shock(df):
    out = df.copy()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.strip()

    if "Recovery" in out.columns:
        out["Recovery"] = pd.to_numeric(out["Recovery"], errors="coerce")

    return out


def filter_validation_variant(df, variant):
    out = df.copy()

    if "Recovery" not in out.columns:
        return out.iloc[0:0].copy()

    out = out[out["Recovery"].notna()].copy()

    if variant.get("exclude_zero", False):
        out = out[out["Recovery"] != 0].copy()

    if variant.get("exclude_twenty", False):
        out = out[out["Recovery"] != 20].copy()

    min_recovery = variant.get("min_recovery", None)
    max_recovery = variant.get("max_recovery", None)

    if min_recovery is not None:
        out = out[out["Recovery"] >= min_recovery].copy()

    if max_recovery is not None:
        out = out[out["Recovery"] <= max_recovery].copy()

    return out


def rank_correlation(x, y):
    temp = pd.DataFrame({"x": x, "y": y}).dropna()

    if len(temp) < 3:
        return np.nan

    if temp["x"].nunique() < 2 or temp["y"].nunique() < 2:
        return np.nan

    return temp["x"].rank(method="average").corr(temp["y"].rank(method="average"))


def mean_difference(group_a, group_b):
    a = pd.to_numeric(group_a, errors="coerce").dropna()
    b = pd.to_numeric(group_b, errors="coerce").dropna()

    if len(a) == 0 or len(b) == 0:
        return np.nan

    return a.mean() - b.mean()


def median_difference(group_a, group_b):
    a = pd.to_numeric(group_a, errors="coerce").dropna()
    b = pd.to_numeric(group_b, errors="coerce").dropna()

    if len(a) == 0 or len(b) == 0:
        return np.nan

    return a.median() - b.median()


def recovery_summary_by_group(df, group_col):
    if df.empty or group_col not in df.columns:
        return pd.DataFrame()

    summary = (
        df
        .groupby(group_col, dropna=False)
        .agg(
            country_count=("Country", "nunique"),
            recovery_mean=("Recovery", "mean"),
            recovery_median=("Recovery", "median"),
            recovery_min=("Recovery", "min"),
            recovery_max=("Recovery", "max"),
            recovery_zero_count=("Recovery", lambda x: int((x == 0).sum())),
            recovery_twenty_count=("Recovery", lambda x: int((x == 20).sum())),
            countries=("Country", lambda x: ";".join(sorted(x.astype(str).unique()))),
        )
        .reset_index()
    )

    return summary


def add_metadata(table, metadata):
    out = table.copy()

    for col, val in metadata.items():
        if col in out.columns:
            out[col] = val
        else:
            out.insert(0, col, val)

    order = list(metadata.keys())
    cols = [col for col in order if col in out.columns]
    cols += [col for col in out.columns if col not in cols]

    return out[cols].copy()


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))

In [4]:
# ------------------------------------------------------
# LOAD INPUTS
# ------------------------------------------------------

profile_country_map = read_csv_if_exists(PROFILE_COUNTRY_MAP_FILE, "profile_country_map", required=True)
profile_run_summary = read_csv_if_exists(PROFILE_RUN_SUMMARY_FILE, "profile_run_summary", required=True)
profile_layer_summary = read_csv_if_exists(PROFILE_LAYER_SUMMARY_FILE, "profile_layer_summary")
profile_pareto_countries = read_csv_if_exists(PROFILE_PARETO_COUNTRIES_FILE, "profile_pareto_countries")

working_main_profile_country_map = read_csv_if_exists(WORKING_MAIN_PROFILE_COUNTRY_MAP_FILE, "working_main_profile_country_map")
working_main_profile_run_summary = read_csv_if_exists(WORKING_MAIN_PROFILE_RUN_SUMMARY_FILE, "working_main_profile_run_summary")
working_main_pareto_countries = read_csv_if_exists(WORKING_MAIN_PARETO_COUNTRIES_FILE, "working_main_pareto_countries")

epsilon_country_summary = read_csv_if_exists(EPSILON_COUNTRY_SUMMARY_FILE, "epsilon_country_summary")
epsilon_run_summary = read_csv_if_exists(EPSILON_RUN_SUMMARY_FILE, "epsilon_run_summary")
working_main_epsilon_country_summary = read_csv_if_exists(WORKING_MAIN_EPSILON_COUNTRY_SUMMARY_FILE, "working_main_epsilon_country_summary")
working_main_epsilon_run_summary = read_csv_if_exists(WORKING_MAIN_EPSILON_RUN_SUMMARY_FILE, "working_main_epsilon_run_summary")

epsilon_margin_country_summary = read_csv_if_exists(EPSILON_MARGIN_COUNTRY_SUMMARY_FILE, "epsilon_margin_country_summary")
epsilon_margin_run_summary = read_csv_if_exists(EPSILON_MARGIN_RUN_SUMMARY_FILE, "epsilon_margin_run_summary")
working_main_epsilon_margin_country_summary = read_csv_if_exists(WORKING_MAIN_EPSILON_MARGIN_COUNTRY_SUMMARY_FILE, "working_main_epsilon_margin_country_summary")
working_main_epsilon_margin_run_summary = read_csv_if_exists(WORKING_MAIN_EPSILON_MARGIN_RUN_SUMMARY_FILE, "working_main_epsilon_margin_run_summary")

pre_poset_snapshot = read_csv_if_exists(PRE_POSET_SNAPSHOT_FILE, "pre_poset_snapshot")

# Clean key types.
for name in [
    "profile_country_map",
    "profile_pareto_countries",
    "working_main_profile_country_map",
    "working_main_pareto_countries",
    "epsilon_country_summary",
    "working_main_epsilon_country_summary",
    "epsilon_margin_country_summary",
    "working_main_epsilon_margin_country_summary",
    "pre_poset_snapshot",
]:
    if name in globals():
        globals()[name] = clean_country_shock(globals()[name])

for df_name in [
    "profile_run_summary",
    "working_main_profile_run_summary",
    "epsilon_run_summary",
    "working_main_epsilon_run_summary",
    "epsilon_margin_run_summary",
    "working_main_epsilon_margin_run_summary",
]:
    df = globals().get(df_name, pd.DataFrame())

    if not df.empty and "shock_id" in df.columns:
        df["shock_id"] = df["shock_id"].astype(str)
        globals()[df_name] = df

input_summary = pd.DataFrame([
    {"input": "profile_country_map", "rows": len(profile_country_map), "columns": len(profile_country_map.columns)},
    {"input": "profile_run_summary", "rows": len(profile_run_summary), "columns": len(profile_run_summary.columns)},
    {"input": "epsilon_country_summary", "rows": len(epsilon_country_summary), "columns": len(epsilon_country_summary.columns)},
    {"input": "epsilon_margin_country_summary", "rows": len(epsilon_margin_country_summary), "columns": len(epsilon_margin_country_summary.columns)},
    {"input": "pre_poset_snapshot", "rows": len(pre_poset_snapshot), "columns": len(pre_poset_snapshot.columns)},
])

input_summary.to_csv(
    DIAGNOSTICS_DIR / "recovery_validation_input_summary.csv",
    index=False,
)

display(input_summary)

Loaded profile_country_map: 349 rows
Loaded profile_run_summary: 12 rows
Loaded profile_layer_summary: 61 rows
Loaded profile_pareto_countries: 120 rows
Loaded working_main_profile_country_map: 57 rows
Loaded working_main_profile_run_summary: 2 rows
Loaded working_main_pareto_countries: 22 rows
Loaded epsilon_country_summary: 1410 rows
Loaded epsilon_run_summary: 48 rows
Loaded working_main_epsilon_country_summary: 342 rows
Loaded working_main_epsilon_run_summary: 12 rows
Loaded epsilon_margin_country_summary: 1410 rows
Loaded epsilon_margin_run_summary: 48 rows
Loaded working_main_epsilon_margin_country_summary: 342 rows
Loaded working_main_epsilon_margin_run_summary: 12 rows
Loaded pre_poset_snapshot: 300 rows


,input,rows,columns
0,profile_country_map,349,46
1,profile_run_summary,12,23
2,epsilon_country_summary,1410,32
3,epsilon_margin_country_summary,1410,32
4,pre_poset_snapshot,300,182


In [5]:
# ------------------------------------------------------
# PROFILE-LEVEL RECOVERY VALIDATION
# ------------------------------------------------------

profile_validation_rows = []
profile_group_summaries = []

if not profile_country_map.empty:
    profile_country_map["shock_id"] = profile_country_map["shock_id"].astype(str)

    for (run_key, set_name, shock_id, profile_levels), run_df in profile_country_map.groupby(
        ["run_key", "set_name", "shock_id", "profile_levels"],
        dropna=False,
    ):
        run_df = run_df.copy()

        if "is_pareto_profile" in run_df.columns:
            run_df["is_pareto_profile"] = run_df["is_pareto_profile"].astype(bool)

        for variant in VALIDATION_VARIANTS:
            variant_name = variant["validation_variant"]
            validation_df = filter_validation_variant(run_df, variant)

            if validation_df.empty:
                continue

            pareto_df = validation_df[validation_df["is_pareto_profile"] == True].copy()
            non_pareto_df = validation_df[validation_df["is_pareto_profile"] == False].copy()

            pareto_minus_non_pareto_mean = mean_difference(
                pareto_df["Recovery"],
                non_pareto_df["Recovery"],
            )

            pareto_minus_non_pareto_median = median_difference(
                pareto_df["Recovery"],
                non_pareto_df["Recovery"],
            )

            layer_corr = rank_correlation(
                validation_df["poset_layer"],
                validation_df["Recovery"],
            ) if "poset_layer" in validation_df.columns else np.nan

            dominates_corr = rank_correlation(
                validation_df["dominates_profile_count"],
                validation_df["Recovery"],
            ) if "dominates_profile_count" in validation_df.columns else np.nan

            dominated_by_corr = rank_correlation(
                validation_df["dominated_by_profile_count"],
                validation_df["Recovery"],
            ) if "dominated_by_profile_count" in validation_df.columns else np.nan

            profile_validation_rows.append({
                "run_key": run_key,
                "set_name": set_name,
                "shock_id": shock_id,
                "profile_levels": profile_levels,
                "validation_variant": variant_name,
                "country_count": validation_df["Country"].nunique(),
                "pareto_country_count": pareto_df["Country"].nunique(),
                "non_pareto_country_count": non_pareto_df["Country"].nunique(),
                "recovery_mean_all": validation_df["Recovery"].mean(),
                "recovery_median_all": validation_df["Recovery"].median(),
                "pareto_recovery_mean": pareto_df["Recovery"].mean(),
                "non_pareto_recovery_mean": non_pareto_df["Recovery"].mean(),
                "pareto_minus_non_pareto_mean": pareto_minus_non_pareto_mean,
                "pareto_recovery_median": pareto_df["Recovery"].median(),
                "non_pareto_recovery_median": non_pareto_df["Recovery"].median(),
                "pareto_minus_non_pareto_median": pareto_minus_non_pareto_median,
                "spearman_poset_layer_vs_recovery": layer_corr,
                "spearman_dominates_count_vs_recovery": dominates_corr,
                "spearman_dominated_by_count_vs_recovery": dominated_by_corr,
                "interpretation_note": "Lower recovery is faster. Negative Pareto-minus-non-Pareto means Pareto countries recovered faster on average.",
            })

            if "poset_layer" in validation_df.columns:
                layer_summary = recovery_summary_by_group(validation_df, "poset_layer")

                if not layer_summary.empty:
                    layer_summary = add_metadata(
                        layer_summary,
                        {
                            "run_key": run_key,
                            "set_name": set_name,
                            "shock_id": shock_id,
                            "profile_levels": profile_levels,
                            "validation_variant": variant_name,
                            "group_type": "profile_poset_layer",
                        },
                    )

                    profile_group_summaries.append(layer_summary)

            pareto_summary = recovery_summary_by_group(validation_df, "is_pareto_profile")

            if not pareto_summary.empty:
                pareto_summary = add_metadata(
                    pareto_summary,
                    {
                        "run_key": run_key,
                        "set_name": set_name,
                        "shock_id": shock_id,
                        "profile_levels": profile_levels,
                        "validation_variant": variant_name,
                        "group_type": "profile_pareto_vs_non_pareto",
                    },
                )

                profile_group_summaries.append(pareto_summary)

profile_recovery_validation_summary = pd.DataFrame(profile_validation_rows)

profile_recovery_group_summaries = (
    pd.concat(profile_group_summaries, ignore_index=True)
    if profile_group_summaries
    else pd.DataFrame()
)

profile_recovery_validation_summary.to_csv(
    PROCESSED_DIR / "profile_recovery_validation_summary.csv",
    index=False,
)

profile_recovery_validation_summary.to_csv(
    DIAGNOSTICS_DIR / "profile_recovery_validation_summary.csv",
    index=False,
)

profile_recovery_group_summaries.to_csv(
    PROCESSED_DIR / "profile_recovery_group_summaries.csv",
    index=False,
)

profile_recovery_group_summaries.to_csv(
    DIAGNOSTICS_DIR / "profile_recovery_group_summaries.csv",
    index=False,
)

display(profile_recovery_validation_summary.head(100))
display(profile_recovery_group_summaries.head(100))

,run_key,set_name,shock_id,profile_levels,validation_variant,country_count,pareto_country_count,non_pareto_country_count,recovery_mean_all,recovery_median_all,pareto_recovery_mean,non_pareto_recovery_mean,pareto_minus_non_pareto_mean,pareto_recovery_median,non_pareto_recovery_median,pareto_minus_non_pareto_median,spearman_poset_layer_vs_recovery,spearman_dominates_count_vs_recovery,spearman_dominated_by_count_vs_recovery,interpretation_note
0,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,25,1,24,5.5600,5.0000,7.0000,5.5000,1.5000,7.0000,5.0000,2.0000,0.1871,-0.0861,0.2174,Lower recovery is faster. Negative Pareto-minu...
1,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,affected_only_exclude_zero,24,1,23,5.7917,5.0000,7.0000,5.7391,1.2609,7.0000,5.0000,2.0000,0.2261,-0.1989,0.2936,Lower recovery is faster. Negative Pareto-minu...
2,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,recovered_only_exclude_20,24,1,23,4.9583,5.0000,7.0000,4.8696,2.1304,7.0000,5.0000,2.0000,0.0783,0.0135,0.1130,Lower recovery is faster. Negative Pareto-minu...
3,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,affected_recovered_only_exclude_0_and_20,23,1,22,5.1739,5.0000,7.0000,5.0909,1.9091,7.0000,5.0000,2.0000,0.1175,-0.1002,0.1948,Lower recovery is faster. Negative Pareto-minu...
4,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,all_recovery_available,25,8,17,5.5600,5.0000,5.0000,5.8235,-0.8235,6.0000,5.0000,1.0000,0.1049,0.0309,0.1647,Lower recovery is faster. Negative Pareto-minu...
5,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,affected_only_exclude_zero,24,8,16,5.7917,5.0000,5.0000,6.1875,-1.1875,6.0000,5.0000,1.0000,0.1782,-0.0538,0.2283,Lower recovery is faster. Negative Pareto-minu...
6,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,recovered_only_exclude_20,24,8,16,4.9583,5.0000,5.0000,4.9375,0.0625,6.0000,5.0000,1.0000,-0.0186,0.1384,0.0518,Lower recovery is faster. Negative Pareto-minu...
7,baseline_6_variables__shock_2007__levels_4,baseline_6_variables,2007,4,affected_recovered_only_exclude_0_and_20,23,8,15,5.1739,5.0000,5.0000,5.2667,-0.2667,6.0000,5.0000,1.0000,0.0594,0.0564,0.1188,Lower recovery is faster. Negative Pareto-minu...
8,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,all_recovery_available,25,8,17,5.5600,5.0000,5.0000,5.8235,-0.8235,6.0000,5.0000,1.0000,0.0260,-0.0969,0.1264,Lower recovery is faster. Negative Pareto-minu...
9,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,affected_only_exclude_zero,24,8,16,5.7917,5.0000,5.0000,6.1875,-1.1875,6.0000,5.0000,1.0000,0.0928,-0.1431,0.2028,Lower recovery is faster. Negative Pareto-minu...


,run_key,set_name,shock_id,profile_levels,validation_variant,group_type,poset_layer,country_count,recovery_mean,recovery_median,recovery_min,recovery_max,recovery_zero_count,recovery_twenty_count,countries,is_pareto_profile
0,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,profile_poset_layer,1.0000,1,7.0000,7.0000,7.0000,7.0000,0,0,DNK,NaN
1,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,profile_poset_layer,2.0000,8,4.7500,5.5000,1.0000,8.0000,0,0,AUT;CZE;EST;IRL;NLD;SVN;SWE;USA,NaN
2,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,profile_poset_layer,3.0000,5,2.8000,2.0000,1.0000,5.0000,0,0,BEL;CAN;GBR;LTU;LUX,NaN
3,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,profile_poset_layer,4.0000,3,6.0000,8.0000,0.0000,10.0000,1,0,FIN;LVA;POL,NaN
4,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,profile_poset_layer,5.0000,4,5.7500,3.5000,1.0000,15.0000,0,0,DEU;HUN;ITA;SVK,NaN
5,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,profile_poset_layer,6.0000,2,5.0000,5.0000,2.0000,8.0000,0,0,ESP;FRA,NaN
6,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,profile_poset_layer,7.0000,2,14.5000,14.5000,9.0000,20.0000,0,1,GRC;PRT,NaN
7,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,profile_pareto_vs_non_pareto,NaN,24,5.5000,5.0000,0.0000,20.0000,1,1,AUT;BEL;CAN;CZE;DEU;ESP;EST;FIN;FRA;GBR;GRC;HU...,False
8,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,all_recovery_available,profile_pareto_vs_non_pareto,NaN,1,7.0000,7.0000,7.0000,7.0000,0,0,DNK,True
9,baseline_6_variables__shock_2007__levels_3,baseline_6_variables,2007,3,affected_only_exclude_zero,profile_poset_layer,1.0000,1,7.0000,7.0000,7.0000,7.0000,0,0,DNK,NaN


In [6]:
# ------------------------------------------------------
# WORKING MAIN PROFILE VALIDATION
# ------------------------------------------------------

working_main_profile_validation_summary = profile_recovery_validation_summary[
    (profile_recovery_validation_summary["set_name"] == MAIN_SET_NAME)
    & (profile_recovery_validation_summary["profile_levels"] == WORKING_MAIN_PROFILE_LEVEL)
].copy()

working_main_profile_group_summaries = profile_recovery_group_summaries[
    (profile_recovery_group_summaries["set_name"] == MAIN_SET_NAME)
    & (profile_recovery_group_summaries["profile_levels"] == WORKING_MAIN_PROFILE_LEVEL)
].copy() if not profile_recovery_group_summaries.empty else pd.DataFrame()

working_main_profile_validation_summary.to_csv(
    PROCESSED_DIR / "working_main_profile_recovery_validation_summary.csv",
    index=False,
)

working_main_profile_group_summaries.to_csv(
    PROCESSED_DIR / "working_main_profile_recovery_group_summaries.csv",
    index=False,
)

display(working_main_profile_validation_summary)
display(working_main_profile_group_summaries.head(100))

,run_key,set_name,shock_id,profile_levels,validation_variant,country_count,pareto_country_count,non_pareto_country_count,recovery_mean_all,recovery_median_all,pareto_recovery_mean,non_pareto_recovery_mean,pareto_minus_non_pareto_mean,pareto_recovery_median,non_pareto_recovery_median,pareto_minus_non_pareto_median,spearman_poset_layer_vs_recovery,spearman_dominates_count_vs_recovery,spearman_dominated_by_count_vs_recovery,interpretation_note
8,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,all_recovery_available,25,8,17,5.5600,5.0000,5.0000,5.8235,-0.8235,6.0000,5.0000,1.0000,0.0260,-0.0969,0.1264,Lower recovery is faster. Negative Pareto-minu...
9,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,affected_only_exclude_zero,24,8,16,5.7917,5.0000,5.0000,6.1875,-1.1875,6.0000,5.0000,1.0000,0.0928,-0.1431,0.2028,Lower recovery is faster. Negative Pareto-minu...
10,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,recovered_only_exclude_20,24,8,16,4.9583,5.0000,5.0000,4.9375,0.0625,6.0000,5.0000,1.0000,-0.1082,0.0047,0.0079,Lower recovery is faster. Negative Pareto-minu...
11,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,affected_recovered_only_exclude_0_and_20,23,8,15,5.1739,5.0000,5.0000,5.2667,-0.2667,6.0000,5.0000,1.0000,-0.0386,-0.0448,0.0893,Lower recovery is faster. Negative Pareto-minu...
20,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,all_recovery_available,32,14,18,1.1562,1.0000,1.0714,1.2222,-0.1508,1.0000,1.0000,0.0000,0.2813,-0.0859,0.2794,Lower recovery is faster. Negative Pareto-minu...
21,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,affected_only_exclude_zero,27,12,15,1.3704,1.0000,1.2500,1.4667,-0.2167,1.0000,1.0000,0.0000,0.3644,-0.2198,0.3931,Lower recovery is faster. Negative Pareto-minu...
22,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,recovered_only_exclude_20,32,14,18,1.1562,1.0000,1.0714,1.2222,-0.1508,1.0000,1.0000,0.0000,0.2813,-0.0859,0.2794,Lower recovery is faster. Negative Pareto-minu...
23,baseline_6_variables__shock_2019__levels_5,baseline_6_variables,2019,5,affected_recovered_only_exclude_0_and_20,27,12,15,1.3704,1.0000,1.2500,1.4667,-0.2167,1.0000,1.0000,0.0000,0.3644,-0.2198,0.3931,Lower recovery is faster. Negative Pareto-minu...


,run_key,set_name,shock_id,profile_levels,validation_variant,group_type,poset_layer,country_count,recovery_mean,recovery_median,recovery_min,recovery_max,recovery_zero_count,recovery_twenty_count,countries,is_pareto_profile
62,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,all_recovery_available,profile_poset_layer,1.0000,8,5.0000,6.0000,1.0000,8.0000,0,0,CAN;DNK;EST;FIN;GBR;LUX;SVN;USA,NaN
63,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,all_recovery_available,profile_poset_layer,2.0000,8,5.5000,5.5000,2.0000,10.0000,0,0,AUT;CZE;ESP;IRL;LTU;LVA;NLD;SWE,NaN
64,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,all_recovery_available,profile_poset_layer,3.0000,5,3.8000,1.0000,0.0000,15.0000,1,0,BEL;FRA;ITA;POL;SVK,NaN
65,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,all_recovery_available,profile_poset_layer,4.0000,3,5.3333,5.0000,2.0000,9.0000,0,0,DEU;HUN;PRT,NaN
66,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,all_recovery_available,profile_poset_layer,5.0000,1,20.0000,20.0000,20.0000,20.0000,0,1,GRC,NaN
67,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,all_recovery_available,profile_pareto_vs_non_pareto,NaN,17,5.8235,5.0000,0.0000,20.0000,1,1,AUT;BEL;CZE;DEU;ESP;FRA;GRC;HUN;IRL;ITA;LTU;LV...,False
68,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,all_recovery_available,profile_pareto_vs_non_pareto,NaN,8,5.0000,6.0000,1.0000,8.0000,0,0,CAN;DNK;EST;FIN;GBR;LUX;SVN;USA,True
69,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,affected_only_exclude_zero,profile_poset_layer,1.0000,8,5.0000,6.0000,1.0000,8.0000,0,0,CAN;DNK;EST;FIN;GBR;LUX;SVN;USA,NaN
70,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,affected_only_exclude_zero,profile_poset_layer,2.0000,8,5.5000,5.5000,2.0000,10.0000,0,0,AUT;CZE;ESP;IRL;LTU;LVA;NLD;SWE,NaN
71,baseline_6_variables__shock_2007__levels_5,baseline_6_variables,2007,5,affected_only_exclude_zero,profile_poset_layer,3.0000,4,4.7500,1.5000,1.0000,15.0000,0,0,BEL;FRA;ITA;SVK,NaN


In [7]:
# ------------------------------------------------------
# EPSILON TOLERANCE RECOVERY VALIDATION
# ------------------------------------------------------

epsilon_validation_rows = []
epsilon_group_summaries = []

if not epsilon_country_summary.empty:
    epsilon_country_summary["shock_id"] = epsilon_country_summary["shock_id"].astype(str)

    for (run_key, set_name, shock_id, epsilon), run_df in epsilon_country_summary.groupby(
        ["run_key", "set_name", "shock_id", "epsilon"],
        dropna=False,
    ):
        run_df = run_df.copy()

        if "is_pareto_frontier" in run_df.columns:
            run_df["is_pareto_frontier"] = run_df["is_pareto_frontier"].astype(bool)

        for variant in VALIDATION_VARIANTS:
            variant_name = variant["validation_variant"]
            validation_df = filter_validation_variant(run_df, variant)

            if validation_df.empty:
                continue

            frontier_df = validation_df[validation_df["is_pareto_frontier"] == True].copy()
            non_frontier_df = validation_df[validation_df["is_pareto_frontier"] == False].copy()

            frontier_minus_non_mean = mean_difference(
                frontier_df["Recovery"],
                non_frontier_df["Recovery"],
            )

            frontier_minus_non_median = median_difference(
                frontier_df["Recovery"],
                non_frontier_df["Recovery"],
            )

            layer_corr = rank_correlation(
                validation_df["epsilon_layer"],
                validation_df["Recovery"],
            ) if "epsilon_layer" in validation_df.columns else np.nan

            dominates_corr = rank_correlation(
                validation_df["dominates_country_count"],
                validation_df["Recovery"],
            ) if "dominates_country_count" in validation_df.columns else np.nan

            dominated_by_corr = rank_correlation(
                validation_df["dominated_by_country_count"],
                validation_df["Recovery"],
            ) if "dominated_by_country_count" in validation_df.columns else np.nan

            epsilon_validation_rows.append({
                "run_key": run_key,
                "set_name": set_name,
                "shock_id": shock_id,
                "epsilon": epsilon,
                "validation_variant": variant_name,
                "country_count": validation_df["Country"].nunique(),
                "frontier_country_count": frontier_df["Country"].nunique(),
                "non_frontier_country_count": non_frontier_df["Country"].nunique(),
                "recovery_mean_all": validation_df["Recovery"].mean(),
                "recovery_median_all": validation_df["Recovery"].median(),
                "frontier_recovery_mean": frontier_df["Recovery"].mean(),
                "non_frontier_recovery_mean": non_frontier_df["Recovery"].mean(),
                "frontier_minus_non_frontier_mean": frontier_minus_non_mean,
                "frontier_recovery_median": frontier_df["Recovery"].median(),
                "non_frontier_recovery_median": non_frontier_df["Recovery"].median(),
                "frontier_minus_non_frontier_median": frontier_minus_non_median,
                "spearman_epsilon_layer_vs_recovery": layer_corr,
                "spearman_dominates_count_vs_recovery": dominates_corr,
                "spearman_dominated_by_count_vs_recovery": dominated_by_corr,
                "interpretation_note": "Lower recovery is faster. Negative frontier-minus-non-frontier means frontier countries recovered faster on average.",
            })

            if "epsilon_layer" in validation_df.columns:
                layer_summary = recovery_summary_by_group(validation_df, "epsilon_layer")

                if not layer_summary.empty:
                    layer_summary = add_metadata(
                        layer_summary,
                        {
                            "run_key": run_key,
                            "set_name": set_name,
                            "shock_id": shock_id,
                            "epsilon": epsilon,
                            "validation_variant": variant_name,
                            "group_type": "epsilon_layer",
                        },
                    )

                    epsilon_group_summaries.append(layer_summary)

            frontier_summary = recovery_summary_by_group(validation_df, "is_pareto_frontier")

            if not frontier_summary.empty:
                frontier_summary = add_metadata(
                    frontier_summary,
                    {
                        "run_key": run_key,
                        "set_name": set_name,
                        "shock_id": shock_id,
                        "epsilon": epsilon,
                        "validation_variant": variant_name,
                        "group_type": "epsilon_frontier_vs_non_frontier",
                    },
                )

                epsilon_group_summaries.append(frontier_summary)

epsilon_recovery_validation_summary = pd.DataFrame(epsilon_validation_rows)

epsilon_recovery_group_summaries = (
    pd.concat(epsilon_group_summaries, ignore_index=True)
    if epsilon_group_summaries
    else pd.DataFrame()
)

epsilon_recovery_validation_summary.to_csv(
    PROCESSED_DIR / "epsilon_recovery_validation_summary.csv",
    index=False,
)

epsilon_recovery_validation_summary.to_csv(
    DIAGNOSTICS_DIR / "epsilon_recovery_validation_summary.csv",
    index=False,
)

epsilon_recovery_group_summaries.to_csv(
    PROCESSED_DIR / "epsilon_recovery_group_summaries.csv",
    index=False,
)

epsilon_recovery_group_summaries.to_csv(
    DIAGNOSTICS_DIR / "epsilon_recovery_group_summaries.csv",
    index=False,
)

display(epsilon_recovery_validation_summary.head(100))

,run_key,set_name,shock_id,epsilon,validation_variant,country_count,frontier_country_count,non_frontier_country_count,recovery_mean_all,recovery_median_all,frontier_recovery_mean,non_frontier_recovery_mean,frontier_minus_non_frontier_mean,frontier_recovery_median,non_frontier_recovery_median,frontier_minus_non_frontier_median,spearman_epsilon_layer_vs_recovery,spearman_dominates_count_vs_recovery,spearman_dominated_by_count_vs_recovery,interpretation_note
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,all_recovery_available,25,13,12,5.5600,5.0000,4.5385,6.6667,-2.1282,5.0000,5.5000,-0.5000,0.2318,0.0405,0.1913,Lower recovery is faster. Negative frontier-mi...
1,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,affected_only_exclude_zero,24,13,11,5.7917,5.0000,4.5385,7.2727,-2.7343,5.0000,6.0000,-1.0000,0.2937,-0.0316,0.2809,Lower recovery is faster. Negative frontier-mi...
2,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,recovered_only_exclude_20,24,13,11,4.9583,5.0000,4.5385,5.4545,-0.9161,5.0000,5.0000,0.0000,0.1375,0.1223,0.0748,Lower recovery is faster. Negative frontier-mi...
3,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,affected_recovered_only_exclude_0_and_20,23,13,10,5.1739,5.0000,4.5385,6.0000,-1.4615,5.0000,5.5000,-0.5000,0.2050,0.0510,0.1710,Lower recovery is faster. Negative frontier-mi...
4,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,all_recovery_available,25,13,12,5.5600,5.0000,4.5385,6.6667,-2.1282,5.0000,5.5000,-0.5000,0.2318,0.0564,0.2165,Lower recovery is faster. Negative frontier-mi...
5,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,affected_only_exclude_zero,24,13,11,5.7917,5.0000,4.5385,7.2727,-2.7343,5.0000,6.0000,-1.0000,0.2937,-0.0137,0.3062,Lower recovery is faster. Negative frontier-mi...
6,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,recovered_only_exclude_20,24,13,11,4.9583,5.0000,4.5385,5.4545,-0.9161,5.0000,5.0000,0.0000,0.1375,0.1401,0.1037,Lower recovery is faster. Negative frontier-mi...
7,baseline_6_variables__shock_2007__eps_0.02,baseline_6_variables,2007,0.0200,affected_recovered_only_exclude_0_and_20,23,13,10,5.1739,5.0000,4.5385,6.0000,-1.4615,5.0000,5.5000,-0.5000,0.2050,0.0712,0.2003,Lower recovery is faster. Negative frontier-mi...
8,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,all_recovery_available,25,10,15,5.5600,5.0000,4.8000,6.0667,-1.2667,5.5000,5.0000,0.5000,0.1670,0.0991,0.1843,Lower recovery is faster. Negative frontier-mi...
9,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,affected_only_exclude_zero,24,10,14,5.7917,5.0000,4.8000,6.5000,-1.7000,5.5000,5.0000,0.5000,0.2051,0.0243,0.2456,Lower recovery is faster. Negative frontier-mi...


In [8]:
# ------------------------------------------------------
# EPSILON-MARGIN RECOVERY VALIDATION
# ------------------------------------------------------

epsilon_margin_validation_rows = []
epsilon_margin_group_summaries = []

if not epsilon_margin_country_summary.empty:
    epsilon_margin_country_summary["shock_id"] = epsilon_margin_country_summary["shock_id"].astype(str)

    for (run_key, set_name, shock_id, epsilon_margin), run_df in epsilon_margin_country_summary.groupby(
        ["run_key", "set_name", "shock_id", "epsilon_margin"],
        dropna=False,
    ):
        run_df = run_df.copy()

        if "is_pareto_frontier" in run_df.columns:
            run_df["is_pareto_frontier"] = run_df["is_pareto_frontier"].astype(bool)

        for variant in VALIDATION_VARIANTS:
            variant_name = variant["validation_variant"]
            validation_df = filter_validation_variant(run_df, variant)

            if validation_df.empty:
                continue

            frontier_df = validation_df[validation_df["is_pareto_frontier"] == True].copy()
            non_frontier_df = validation_df[validation_df["is_pareto_frontier"] == False].copy()

            frontier_minus_non_mean = mean_difference(
                frontier_df["Recovery"],
                non_frontier_df["Recovery"],
            )

            frontier_minus_non_median = median_difference(
                frontier_df["Recovery"],
                non_frontier_df["Recovery"],
            )

            layer_corr = rank_correlation(
                validation_df["epsilon_margin_layer"],
                validation_df["Recovery"],
            ) if "epsilon_margin_layer" in validation_df.columns else np.nan

            dominates_corr = rank_correlation(
                validation_df["dominates_country_count"],
                validation_df["Recovery"],
            ) if "dominates_country_count" in validation_df.columns else np.nan

            dominated_by_corr = rank_correlation(
                validation_df["dominated_by_country_count"],
                validation_df["Recovery"],
            ) if "dominated_by_country_count" in validation_df.columns else np.nan

            epsilon_margin_validation_rows.append({
                "run_key": run_key,
                "set_name": set_name,
                "shock_id": shock_id,
                "epsilon_margin": epsilon_margin,
                "validation_variant": variant_name,
                "country_count": validation_df["Country"].nunique(),
                "frontier_country_count": frontier_df["Country"].nunique(),
                "non_frontier_country_count": non_frontier_df["Country"].nunique(),
                "recovery_mean_all": validation_df["Recovery"].mean(),
                "recovery_median_all": validation_df["Recovery"].median(),
                "frontier_recovery_mean": frontier_df["Recovery"].mean(),
                "non_frontier_recovery_mean": non_frontier_df["Recovery"].mean(),
                "frontier_minus_non_frontier_mean": frontier_minus_non_mean,
                "frontier_recovery_median": frontier_df["Recovery"].median(),
                "non_frontier_recovery_median": non_frontier_df["Recovery"].median(),
                "frontier_minus_non_frontier_median": frontier_minus_non_median,
                "spearman_epsilon_margin_layer_vs_recovery": layer_corr,
                "spearman_dominates_count_vs_recovery": dominates_corr,
                "spearman_dominated_by_count_vs_recovery": dominated_by_corr,
                "interpretation_note": "Lower recovery is faster. Negative frontier-minus-non-frontier means frontier countries recovered faster on average.",
            })

            if "epsilon_margin_layer" in validation_df.columns:
                layer_summary = recovery_summary_by_group(validation_df, "epsilon_margin_layer")

                if not layer_summary.empty:
                    layer_summary = add_metadata(
                        layer_summary,
                        {
                            "run_key": run_key,
                            "set_name": set_name,
                            "shock_id": shock_id,
                            "epsilon_margin": epsilon_margin,
                            "validation_variant": variant_name,
                            "group_type": "epsilon_margin_layer",
                        },
                    )

                    epsilon_margin_group_summaries.append(layer_summary)

            frontier_summary = recovery_summary_by_group(validation_df, "is_pareto_frontier")

            if not frontier_summary.empty:
                frontier_summary = add_metadata(
                    frontier_summary,
                    {
                        "run_key": run_key,
                        "set_name": set_name,
                        "shock_id": shock_id,
                        "epsilon_margin": epsilon_margin,
                        "validation_variant": variant_name,
                        "group_type": "epsilon_margin_frontier_vs_non_frontier",
                    },
                )

                epsilon_margin_group_summaries.append(frontier_summary)

epsilon_margin_recovery_validation_summary = pd.DataFrame(epsilon_margin_validation_rows)

epsilon_margin_recovery_group_summaries = (
    pd.concat(epsilon_margin_group_summaries, ignore_index=True)
    if epsilon_margin_group_summaries
    else pd.DataFrame()
)

epsilon_margin_recovery_validation_summary.to_csv(
    PROCESSED_DIR / "epsilon_margin_recovery_validation_summary.csv",
    index=False,
)

epsilon_margin_recovery_validation_summary.to_csv(
    DIAGNOSTICS_DIR / "epsilon_margin_recovery_validation_summary.csv",
    index=False,
)

epsilon_margin_recovery_group_summaries.to_csv(
    PROCESSED_DIR / "epsilon_margin_recovery_group_summaries.csv",
    index=False,
)

epsilon_margin_recovery_group_summaries.to_csv(
    DIAGNOSTICS_DIR / "epsilon_margin_recovery_group_summaries.csv",
    index=False,
)

display(epsilon_margin_recovery_validation_summary.head(100))

,run_key,set_name,shock_id,epsilon_margin,validation_variant,country_count,frontier_country_count,non_frontier_country_count,recovery_mean_all,recovery_median_all,frontier_recovery_mean,non_frontier_recovery_mean,frontier_minus_non_frontier_mean,frontier_recovery_median,non_frontier_recovery_median,frontier_minus_non_frontier_median,spearman_epsilon_margin_layer_vs_recovery,spearman_dominates_count_vs_recovery,spearman_dominated_by_count_vs_recovery,interpretation_note
0,baseline_6_variables__shock_2007__margin_0.00,baseline_6_variables,2007,0.0000,all_recovery_available,25,13,12,5.5600,5.0000,4.5385,6.6667,-2.1282,5.0000,5.5000,-0.5000,0.2318,0.0405,0.1913,Lower recovery is faster. Negative frontier-mi...
1,baseline_6_variables__shock_2007__margin_0.00,baseline_6_variables,2007,0.0000,affected_only_exclude_zero,24,13,11,5.7917,5.0000,4.5385,7.2727,-2.7343,5.0000,6.0000,-1.0000,0.2937,-0.0316,0.2809,Lower recovery is faster. Negative frontier-mi...
2,baseline_6_variables__shock_2007__margin_0.00,baseline_6_variables,2007,0.0000,recovered_only_exclude_20,24,13,11,4.9583,5.0000,4.5385,5.4545,-0.9161,5.0000,5.0000,0.0000,0.1375,0.1223,0.0748,Lower recovery is faster. Negative frontier-mi...
3,baseline_6_variables__shock_2007__margin_0.00,baseline_6_variables,2007,0.0000,affected_recovered_only_exclude_0_and_20,23,13,10,5.1739,5.0000,4.5385,6.0000,-1.4615,5.0000,5.5000,-0.5000,0.2050,0.0510,0.1710,Lower recovery is faster. Negative frontier-mi...
4,baseline_6_variables__shock_2007__margin_0.02,baseline_6_variables,2007,0.0200,all_recovery_available,25,13,12,5.5600,5.0000,4.5385,6.6667,-2.1282,5.0000,5.5000,-0.5000,0.2318,0.0405,0.1913,Lower recovery is faster. Negative frontier-mi...
5,baseline_6_variables__shock_2007__margin_0.02,baseline_6_variables,2007,0.0200,affected_only_exclude_zero,24,13,11,5.7917,5.0000,4.5385,7.2727,-2.7343,5.0000,6.0000,-1.0000,0.2937,-0.0316,0.2809,Lower recovery is faster. Negative frontier-mi...
6,baseline_6_variables__shock_2007__margin_0.02,baseline_6_variables,2007,0.0200,recovered_only_exclude_20,24,13,11,4.9583,5.0000,4.5385,5.4545,-0.9161,5.0000,5.0000,0.0000,0.1375,0.1223,0.0748,Lower recovery is faster. Negative frontier-mi...
7,baseline_6_variables__shock_2007__margin_0.02,baseline_6_variables,2007,0.0200,affected_recovered_only_exclude_0_and_20,23,13,10,5.1739,5.0000,4.5385,6.0000,-1.4615,5.0000,5.5000,-0.5000,0.2050,0.0510,0.1710,Lower recovery is faster. Negative frontier-mi...
8,baseline_6_variables__shock_2007__margin_0.05,baseline_6_variables,2007,0.0500,all_recovery_available,25,13,12,5.5600,5.0000,4.5385,6.6667,-2.1282,5.0000,5.5000,-0.5000,0.2318,0.0405,0.1913,Lower recovery is faster. Negative frontier-mi...
9,baseline_6_variables__shock_2007__margin_0.05,baseline_6_variables,2007,0.0500,affected_only_exclude_zero,24,13,11,5.7917,5.0000,4.5385,7.2727,-2.7343,5.0000,6.0000,-1.0000,0.2937,-0.0316,0.2809,Lower recovery is faster. Negative frontier-mi...


In [9]:
# ------------------------------------------------------
# WORKING MAIN EPSILON AND EPSILON-MARGIN VALIDATION
# ------------------------------------------------------

working_main_epsilon_validation_summary = epsilon_recovery_validation_summary[
    (epsilon_recovery_validation_summary["set_name"] == MAIN_SET_NAME)
    & (epsilon_recovery_validation_summary["epsilon"].isin(EPSILON_VALUES_FOR_REVIEW))
].copy() if not epsilon_recovery_validation_summary.empty else pd.DataFrame()

working_main_epsilon_margin_validation_summary = epsilon_margin_recovery_validation_summary[
    (epsilon_margin_recovery_validation_summary["set_name"] == MAIN_SET_NAME)
    & (epsilon_margin_recovery_validation_summary["epsilon_margin"].isin(EPSILON_MARGIN_VALUES_FOR_REVIEW))
].copy() if not epsilon_margin_recovery_validation_summary.empty else pd.DataFrame()

working_main_epsilon_validation_summary.to_csv(
    PROCESSED_DIR / "working_main_epsilon_recovery_validation_summary.csv",
    index=False,
)

working_main_epsilon_margin_validation_summary.to_csv(
    PROCESSED_DIR / "working_main_epsilon_margin_recovery_validation_summary.csv",
    index=False,
)

display(working_main_epsilon_validation_summary)
display(working_main_epsilon_margin_validation_summary)

,run_key,set_name,shock_id,epsilon,validation_variant,country_count,frontier_country_count,non_frontier_country_count,recovery_mean_all,recovery_median_all,frontier_recovery_mean,non_frontier_recovery_mean,frontier_minus_non_frontier_mean,frontier_recovery_median,non_frontier_recovery_median,frontier_minus_non_frontier_median,spearman_epsilon_layer_vs_recovery,spearman_dominates_count_vs_recovery,spearman_dominated_by_count_vs_recovery,interpretation_note
0,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,all_recovery_available,25,13,12,5.5600,5.0000,4.5385,6.6667,-2.1282,5.0000,5.5000,-0.5000,0.2318,0.0405,0.1913,Lower recovery is faster. Negative frontier-mi...
1,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,affected_only_exclude_zero,24,13,11,5.7917,5.0000,4.5385,7.2727,-2.7343,5.0000,6.0000,-1.0000,0.2937,-0.0316,0.2809,Lower recovery is faster. Negative frontier-mi...
2,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,recovered_only_exclude_20,24,13,11,4.9583,5.0000,4.5385,5.4545,-0.9161,5.0000,5.0000,0.0000,0.1375,0.1223,0.0748,Lower recovery is faster. Negative frontier-mi...
3,baseline_6_variables__shock_2007__eps_0.00,baseline_6_variables,2007,0.0000,affected_recovered_only_exclude_0_and_20,23,13,10,5.1739,5.0000,4.5385,6.0000,-1.4615,5.0000,5.5000,-0.5000,0.2050,0.0510,0.1710,Lower recovery is faster. Negative frontier-mi...
8,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,all_recovery_available,25,10,15,5.5600,5.0000,4.8000,6.0667,-1.2667,5.5000,5.0000,0.5000,0.1670,0.0991,0.1843,Lower recovery is faster. Negative frontier-mi...
9,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,affected_only_exclude_zero,24,10,14,5.7917,5.0000,4.8000,6.5000,-1.7000,5.5000,5.0000,0.5000,0.2051,0.0243,0.2456,Lower recovery is faster. Negative frontier-mi...
10,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,recovered_only_exclude_20,24,10,14,4.9583,5.0000,4.8000,5.0714,-0.2714,5.5000,5.0000,0.5000,0.0657,0.1972,0.0716,Lower recovery is faster. Negative frontier-mi...
11,baseline_6_variables__shock_2007__eps_0.05,baseline_6_variables,2007,0.0500,affected_recovered_only_exclude_0_and_20,23,10,13,5.1739,5.0000,4.8000,5.4615,-0.6615,5.5000,5.0000,0.5000,0.1064,0.1245,0.1358,Lower recovery is faster. Negative frontier-mi...
12,baseline_6_variables__shock_2007__eps_0.10,baseline_6_variables,2007,0.1000,all_recovery_available,25,9,16,5.5600,5.0000,4.4444,6.1875,-1.7431,5.0000,5.0000,0.0000,0.2027,0.0236,0.2116,Lower recovery is faster. Negative frontier-mi...
13,baseline_6_variables__shock_2007__eps_0.10,baseline_6_variables,2007,0.1000,affected_only_exclude_zero,24,9,15,5.7917,5.0000,4.4444,6.6000,-2.1556,5.0000,5.0000,0.0000,0.2111,-0.0672,0.2734,Lower recovery is faster. Negative frontier-mi...


,run_key,set_name,shock_id,epsilon_margin,validation_variant,country_count,frontier_country_count,non_frontier_country_count,recovery_mean_all,recovery_median_all,frontier_recovery_mean,non_frontier_recovery_mean,frontier_minus_non_frontier_mean,frontier_recovery_median,non_frontier_recovery_median,frontier_minus_non_frontier_median,spearman_epsilon_margin_layer_vs_recovery,spearman_dominates_count_vs_recovery,spearman_dominated_by_count_vs_recovery,interpretation_note
0,baseline_6_variables__shock_2007__margin_0.00,baseline_6_variables,2007,0.0000,all_recovery_available,25,13,12,5.5600,5.0000,4.5385,6.6667,-2.1282,5.0000,5.5000,-0.5000,0.2318,0.0405,0.1913,Lower recovery is faster. Negative frontier-mi...
1,baseline_6_variables__shock_2007__margin_0.00,baseline_6_variables,2007,0.0000,affected_only_exclude_zero,24,13,11,5.7917,5.0000,4.5385,7.2727,-2.7343,5.0000,6.0000,-1.0000,0.2937,-0.0316,0.2809,Lower recovery is faster. Negative frontier-mi...
2,baseline_6_variables__shock_2007__margin_0.00,baseline_6_variables,2007,0.0000,recovered_only_exclude_20,24,13,11,4.9583,5.0000,4.5385,5.4545,-0.9161,5.0000,5.0000,0.0000,0.1375,0.1223,0.0748,Lower recovery is faster. Negative frontier-mi...
3,baseline_6_variables__shock_2007__margin_0.00,baseline_6_variables,2007,0.0000,affected_recovered_only_exclude_0_and_20,23,13,10,5.1739,5.0000,4.5385,6.0000,-1.4615,5.0000,5.5000,-0.5000,0.2050,0.0510,0.1710,Lower recovery is faster. Negative frontier-mi...
8,baseline_6_variables__shock_2007__margin_0.05,baseline_6_variables,2007,0.0500,all_recovery_available,25,13,12,5.5600,5.0000,4.5385,6.6667,-2.1282,5.0000,5.5000,-0.5000,0.2318,0.0405,0.1913,Lower recovery is faster. Negative frontier-mi...
9,baseline_6_variables__shock_2007__margin_0.05,baseline_6_variables,2007,0.0500,affected_only_exclude_zero,24,13,11,5.7917,5.0000,4.5385,7.2727,-2.7343,5.0000,6.0000,-1.0000,0.2937,-0.0316,0.2809,Lower recovery is faster. Negative frontier-mi...
10,baseline_6_variables__shock_2007__margin_0.05,baseline_6_variables,2007,0.0500,recovered_only_exclude_20,24,13,11,4.9583,5.0000,4.5385,5.4545,-0.9161,5.0000,5.0000,0.0000,0.1375,0.1223,0.0748,Lower recovery is faster. Negative frontier-mi...
11,baseline_6_variables__shock_2007__margin_0.05,baseline_6_variables,2007,0.0500,affected_recovered_only_exclude_0_and_20,23,13,10,5.1739,5.0000,4.5385,6.0000,-1.4615,5.0000,5.5000,-0.5000,0.2050,0.0510,0.1710,Lower recovery is faster. Negative frontier-mi...
12,baseline_6_variables__shock_2007__margin_0.10,baseline_6_variables,2007,0.1000,all_recovery_available,25,13,12,5.5600,5.0000,4.5385,6.6667,-2.1282,5.0000,5.5000,-0.5000,0.2318,0.0405,0.1913,Lower recovery is faster. Negative frontier-mi...
13,baseline_6_variables__shock_2007__margin_0.10,baseline_6_variables,2007,0.1000,affected_only_exclude_zero,24,13,11,5.7917,5.0000,4.5385,7.2727,-2.7343,5.0000,6.0000,-1.0000,0.2937,-0.0316,0.2809,Lower recovery is faster. Negative frontier-mi...


In [10]:
# ------------------------------------------------------
# PROFILE-LEVEL COUNTRY INTERPRETATION TABLES
# ------------------------------------------------------

# Countries with strong structure but slow recovery, and weaker structure but fast recovery.
# This is descriptive only.

interpretation_rows = []

if not working_main_profile_country_map.empty:
    wm = working_main_profile_country_map.copy()
    wm = wm[
        (wm["set_name"] == MAIN_SET_NAME)
        & (wm["profile_levels"] == WORKING_MAIN_PROFILE_LEVEL)
        & (wm["Recovery"].notna())
    ].copy()

    for shock_id, group in wm.groupby("shock_id"):
        if group.empty:
            continue

        recovery_median = group["Recovery"].median()
        layer_median = group["poset_layer"].median()

        strong_slow = group[
            (group["poset_layer"] <= layer_median)
            & (group["Recovery"] > recovery_median)
        ].copy()

        weak_fast = group[
            (group["poset_layer"] > layer_median)
            & (group["Recovery"] <= recovery_median)
        ].copy()

        for _, row in strong_slow.iterrows():
            interpretation_rows.append({
                "shock_id": shock_id,
                "case_type": "stronger_structure_but_slower_recovery",
                "Country": row["Country"],
                "country_name": row.get("country_name", row["Country"]),
                "profile_id": row.get("profile_id", ""),
                "profile_code": row.get("profile_code", ""),
                "poset_layer": row.get("poset_layer", np.nan),
                "Recovery": row.get("Recovery", np.nan),
                "note": "Potential mismatch case for discussion; not an error.",
            })

        for _, row in weak_fast.iterrows():
            interpretation_rows.append({
                "shock_id": shock_id,
                "case_type": "weaker_structure_but_faster_recovery",
                "Country": row["Country"],
                "country_name": row.get("country_name", row["Country"]),
                "profile_id": row.get("profile_id", ""),
                "profile_code": row.get("profile_code", ""),
                "poset_layer": row.get("poset_layer", np.nan),
                "Recovery": row.get("Recovery", np.nan),
                "note": "Potential mismatch case for discussion; not an error.",
            })

recovery_interpretation_cases = pd.DataFrame(interpretation_rows)

recovery_interpretation_cases.to_csv(
    PROCESSED_DIR / "recovery_interpretation_cases.csv",
    index=False,
)

recovery_interpretation_cases.to_csv(
    DIAGNOSTICS_DIR / "recovery_interpretation_cases.csv",
    index=False,
)

display(recovery_interpretation_cases.head(100))

,shock_id,case_type,Country,country_name,profile_id,profile_code,poset_layer,Recovery,note
0,2007,stronger_structure_but_slower_recovery,DNK,Denmark,P001,4-5-5-4-5-5,1,7.0000,Potential mismatch case for discussion; not an...
1,2007,stronger_structure_but_slower_recovery,ESP,Spain,P019,4-1-3-3-1-2,2,8.0000,Potential mismatch case for discussion; not an...
2,2007,stronger_structure_but_slower_recovery,EST,Estonia,P002,5-5-2-5-5-4,1,8.0000,Potential mismatch case for discussion; not an...
3,2007,stronger_structure_but_slower_recovery,FIN,Finland,P009,3-2-5-5-3-3,1,8.0000,Potential mismatch case for discussion; not an...
4,2007,stronger_structure_but_slower_recovery,IRL,Ireland,P012,4-4-2-4-1-5,2,6.0000,Potential mismatch case for discussion; not an...
5,2007,stronger_structure_but_slower_recovery,LVA,Latvia,P017,5-3-1-2-2-2,2,10.0000,Potential mismatch case for discussion; not an...
6,2007,stronger_structure_but_slower_recovery,NLD,Netherlands,P003,3-5-4-4-4-4,2,6.0000,Potential mismatch case for discussion; not an...
7,2007,stronger_structure_but_slower_recovery,SVN,Slovenia,P008,5-4-3-2-4-3,1,8.0000,Potential mismatch case for discussion; not an...
8,2007,weaker_structure_but_faster_recovery,BEL,Belgium,P016,1-2-4-4-1-4,3,1.0000,Potential mismatch case for discussion; not an...
9,2007,weaker_structure_but_faster_recovery,DEU,Germany,P018,2-1-4-2-3-3,4,2.0000,Potential mismatch case for discussion; not an...


In [11]:
# ------------------------------------------------------
# COMPACT RECOVERY VALIDATION TAKEAWAY TABLE
# ------------------------------------------------------

takeaway_rows = []

# Working main profile takeaway.
if not working_main_profile_validation_summary.empty:
    for _, row in working_main_profile_validation_summary.iterrows():
        if row["validation_variant"] not in ["all_recovery_available", "affected_recovered_only_exclude_0_and_20"]:
            continue

        takeaway_rows.append({
            "method": "profile_poset",
            "set_name": row["set_name"],
            "shock_id": row["shock_id"],
            "setting": f"{int(row['profile_levels'])}_levels",
            "validation_variant": row["validation_variant"],
            "country_count": row["country_count"],
            "frontier_or_pareto_count": row["pareto_country_count"],
            "frontier_minus_non_frontier_mean_recovery": row["pareto_minus_non_pareto_mean"],
            "frontier_minus_non_frontier_median_recovery": row["pareto_minus_non_pareto_median"],
            "layer_vs_recovery_spearman": row["spearman_poset_layer_vs_recovery"],
            "interpretation": "Negative mean/median difference means Pareto countries recovered faster. Positive layer correlation means worse layers recover slower.",
        })

# Working main epsilon margin takeaway.
if not working_main_epsilon_margin_validation_summary.empty:
    for _, row in working_main_epsilon_margin_validation_summary.iterrows():
        if row["validation_variant"] not in ["all_recovery_available", "affected_recovered_only_exclude_0_and_20"]:
            continue

        if row["epsilon_margin"] not in [0.0, 0.05, 0.10]:
            continue

        takeaway_rows.append({
            "method": "epsilon_margin",
            "set_name": row["set_name"],
            "shock_id": row["shock_id"],
            "setting": f"margin_{row['epsilon_margin']:.2f}",
            "validation_variant": row["validation_variant"],
            "country_count": row["country_count"],
            "frontier_or_pareto_count": row["frontier_country_count"],
            "frontier_minus_non_frontier_mean_recovery": row["frontier_minus_non_frontier_mean"],
            "frontier_minus_non_frontier_median_recovery": row["frontier_minus_non_frontier_median"],
            "layer_vs_recovery_spearman": row["spearman_epsilon_margin_layer_vs_recovery"],
            "interpretation": "Negative mean/median difference means frontier countries recovered faster. Positive layer correlation means worse layers recover slower.",
        })

recovery_validation_takeaway_table = pd.DataFrame(takeaway_rows)

recovery_validation_takeaway_table.to_csv(
    PROCESSED_DIR / "recovery_validation_takeaway_table.csv",
    index=False,
)

recovery_validation_takeaway_table.to_csv(
    DIAGNOSTICS_DIR / "recovery_validation_takeaway_table.csv",
    index=False,
)

display(recovery_validation_takeaway_table)

,method,set_name,shock_id,setting,validation_variant,country_count,frontier_or_pareto_count,frontier_minus_non_frontier_mean_recovery,frontier_minus_non_frontier_median_recovery,layer_vs_recovery_spearman,interpretation
0,profile_poset,baseline_6_variables,2007,5_levels,all_recovery_available,25,8,-0.8235,1.0000,0.0260,Negative mean/median difference means Pareto c...
1,profile_poset,baseline_6_variables,2007,5_levels,affected_recovered_only_exclude_0_and_20,23,8,-0.2667,1.0000,-0.0386,Negative mean/median difference means Pareto c...
2,profile_poset,baseline_6_variables,2019,5_levels,all_recovery_available,32,14,-0.1508,0.0000,0.2813,Negative mean/median difference means Pareto c...
3,profile_poset,baseline_6_variables,2019,5_levels,affected_recovered_only_exclude_0_and_20,27,12,-0.2167,0.0000,0.3644,Negative mean/median difference means Pareto c...
4,epsilon_margin,baseline_6_variables,2007,margin_0.00,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...
5,epsilon_margin,baseline_6_variables,2007,margin_0.00,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...
6,epsilon_margin,baseline_6_variables,2007,margin_0.05,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...
7,epsilon_margin,baseline_6_variables,2007,margin_0.05,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...
8,epsilon_margin,baseline_6_variables,2007,margin_0.10,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...
9,epsilon_margin,baseline_6_variables,2007,margin_0.10,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...


In [12]:
# ------------------------------------------------------
# RECOVERY VALIDATION FIGURES
# ------------------------------------------------------

# Figure 1: Working main profile recovery by layer.
if not working_main_profile_country_map.empty:
    wm = working_main_profile_country_map.copy()
    wm = wm[
        (wm["set_name"] == MAIN_SET_NAME)
        & (wm["profile_levels"] == WORKING_MAIN_PROFILE_LEVEL)
        & (wm["Recovery"].notna())
    ].copy()

    for shock_id, group in wm.groupby("shock_id"):
        group = group.sort_values("poset_layer").copy()

        if group.empty:
            continue

        layer_stats = (
            group
            .groupby("poset_layer")
            .agg(recovery_median=("Recovery", "median"), country_count=("Country", "nunique"))
            .reset_index()
            .sort_values("poset_layer")
        )

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(layer_stats["poset_layer"], layer_stats["recovery_median"], marker="o")
        ax.set_title(f"Median recovery by profile POSet layer — shock {shock_id}")
        ax.set_xlabel("POSet layer, 1 = Pareto/nondominated")
        ax.set_ylabel("Median recovery years")
        ax.grid(alpha=0.25)
        fig.tight_layout()

        path = FIGURES_DIR / f"working_main_profile_recovery_by_layer_shock_{shock_id}.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)

# Figure 2: Working main Pareto vs non-Pareto mean recovery.
if not working_main_profile_validation_summary.empty:
    plot_df = working_main_profile_validation_summary[
        working_main_profile_validation_summary["validation_variant"] == "all_recovery_available"
    ].copy()

    if not plot_df.empty:
        for shock_id, group in plot_df.groupby("shock_id"):
            fig, ax = plt.subplots(figsize=(7, 5))

            labels = ["Pareto", "Non-Pareto"]
            values = [
                group["pareto_recovery_mean"].iloc[0],
                group["non_pareto_recovery_mean"].iloc[0],
            ]

            ax.bar(labels, values)
            ax.set_title(f"Mean recovery: Pareto vs non-Pareto — shock {shock_id}")
            ax.set_ylabel("Mean recovery years")
            ax.grid(axis="y", alpha=0.25)
            fig.tight_layout()

            path = FIGURES_DIR / f"working_main_pareto_vs_non_pareto_recovery_shock_{shock_id}.png"
            fig.savefig(path, dpi=200, bbox_inches="tight")
            plt.close(fig)

# Figure 3: Epsilon-margin frontier vs non-frontier at margin 0.05.
if not working_main_epsilon_margin_validation_summary.empty:
    plot_df = working_main_epsilon_margin_validation_summary[
        (working_main_epsilon_margin_validation_summary["validation_variant"] == "all_recovery_available")
        & (np.isclose(working_main_epsilon_margin_validation_summary["epsilon_margin"], 0.05))
    ].copy()

    if not plot_df.empty:
        for shock_id, group in plot_df.groupby("shock_id"):
            fig, ax = plt.subplots(figsize=(7, 5))

            labels = ["Frontier", "Non-frontier"]
            values = [
                group["frontier_recovery_mean"].iloc[0],
                group["non_frontier_recovery_mean"].iloc[0],
            ]

            ax.bar(labels, values)
            ax.set_title(f"Mean recovery: epsilon-margin frontier vs non-frontier — shock {shock_id}")
            ax.set_ylabel("Mean recovery years")
            ax.grid(axis="y", alpha=0.25)
            fig.tight_layout()

            path = FIGURES_DIR / f"epsilon_margin_frontier_vs_non_frontier_recovery_shock_{shock_id}.png"
            fig.savefig(path, dpi=200, bbox_inches="tight")
            plt.close(fig)

print("Figures created in:")
print(FIGURES_DIR.resolve())

Figures created in:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Recovery_Validation


In [13]:
# ------------------------------------------------------
# DATA DICTIONARY AND METHODOLOGICAL NOTES
# ------------------------------------------------------

recovery_validation_data_dictionary = pd.DataFrame([
    {
        "table": "profile_recovery_validation_summary.csv",
        "column": "pareto_minus_non_pareto_mean",
        "description": "Mean recovery of Pareto countries minus mean recovery of non-Pareto countries. Negative means Pareto countries recovered faster on average.",
    },
    {
        "table": "profile_recovery_validation_summary.csv",
        "column": "spearman_poset_layer_vs_recovery",
        "description": "Rank correlation between POSet layer and recovery. Positive means lower/worse layers tend to recover slower.",
    },
    {
        "table": "epsilon_margin_recovery_validation_summary.csv",
        "column": "frontier_minus_non_frontier_mean",
        "description": "Mean recovery of epsilon-margin frontier countries minus non-frontier countries. Negative means frontier countries recovered faster.",
    },
    {
        "table": "recovery_validation_takeaway_table.csv",
        "column": "validation_variant",
        "description": "Recovery sample treatment: all recovery, exclude zero, exclude 20, or exclude both special cases.",
    },
    {
        "table": "recovery_interpretation_cases.csv",
        "column": "case_type",
        "description": "Descriptive mismatch cases: strong structure but slower recovery, or weaker structure but faster recovery.",
    },
])

recovery_validation_data_dictionary.to_csv(
    PROCESSED_DIR / "recovery_validation_data_dictionary.csv",
    index=False,
)

methodological_notes = pd.DataFrame([
    {
        "topic": "No leakage",
        "note": "Recovery is not used in any POSet construction. It is used only after POSet outputs are created.",
    },
    {
        "topic": "No causality",
        "note": "Recovery validation is associational and descriptive, not causal evidence.",
    },
    {
        "topic": "Special recovery values",
        "note": "Recovery=0 means no negative growth in the shock window; Recovery=20 means not recovered by available data. Multiple validation variants are reported.",
    },
    {
        "topic": "Interpretation of differences",
        "note": "Negative frontier-minus-non-frontier recovery means structurally stronger countries recovered faster on average.",
    },
    {
        "topic": "Interpretation of layer correlation",
        "note": "Positive layer-recovery correlation means lower/worse POSet layers tend to have slower recovery.",
    },
    {
        "topic": "Mismatch cases",
        "note": "Countries with strong structure but slow recovery, or weak structure but fast recovery, should be discussed as structural/outcome mismatches rather than errors.",
    },
])

methodological_notes.to_csv(
    DIAGNOSTICS_DIR / "recovery_validation_methodological_notes.csv",
    index=False,
)

display(recovery_validation_data_dictionary)
display(methodological_notes)

,table,column,description
0,profile_recovery_validation_summary.csv,pareto_minus_non_pareto_mean,Mean recovery of Pareto countries minus mean r...
1,profile_recovery_validation_summary.csv,spearman_poset_layer_vs_recovery,Rank correlation between POSet layer and recov...
2,epsilon_margin_recovery_validation_summary.csv,frontier_minus_non_frontier_mean,Mean recovery of epsilon-margin frontier count...
3,recovery_validation_takeaway_table.csv,validation_variant,"Recovery sample treatment: all recovery, exclu..."
4,recovery_interpretation_cases.csv,case_type,Descriptive mismatch cases: strong structure b...


,topic,note
0,No leakage,Recovery is not used in any POSet construction...
1,No causality,Recovery validation is associational and descr...
2,Special recovery values,Recovery=0 means no negative growth in the sho...
3,Interpretation of differences,Negative frontier-minus-non-frontier recovery ...
4,Interpretation of layer correlation,Positive layer-recovery correlation means lowe...
5,Mismatch cases,Countries with strong structure but slow recov...


In [14]:
# ------------------------------------------------------
# CREATE RECOVERY VALIDATION AUDIT WORKBOOK
# ------------------------------------------------------

RECOVERY_VALIDATION_AUDIT_XLSX = AUDIT_DIR / "10_Recovery_Validation_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_input_summary",
        "description": "Input tables loaded for recovery validation.",
        "path": DIAGNOSTICS_DIR / "recovery_validation_input_summary.csv",
    },
    {
        "sheet_name": "02_profile_validation",
        "description": "Profile POSet recovery validation summary.",
        "path": PROCESSED_DIR / "profile_recovery_validation_summary.csv",
    },
    {
        "sheet_name": "03_profile_groups",
        "description": "Profile POSet recovery group summaries.",
        "path": PROCESSED_DIR / "profile_recovery_group_summaries.csv",
    },
    {
        "sheet_name": "04_working_profile",
        "description": "Working main profile validation summary.",
        "path": PROCESSED_DIR / "working_main_profile_recovery_validation_summary.csv",
    },
    {
        "sheet_name": "05_epsilon_validation",
        "description": "Epsilon tolerance recovery validation summary.",
        "path": PROCESSED_DIR / "epsilon_recovery_validation_summary.csv",
    },
    {
        "sheet_name": "06_margin_validation",
        "description": "Epsilon-margin recovery validation summary.",
        "path": PROCESSED_DIR / "epsilon_margin_recovery_validation_summary.csv",
    },
    {
        "sheet_name": "07_working_margin",
        "description": "Working main epsilon-margin validation summary.",
        "path": PROCESSED_DIR / "working_main_epsilon_margin_recovery_validation_summary.csv",
    },
    {
        "sheet_name": "08_takeaways",
        "description": "Compact recovery validation takeaway table.",
        "path": PROCESSED_DIR / "recovery_validation_takeaway_table.csv",
    },
    {
        "sheet_name": "09_cases",
        "description": "Descriptive mismatch cases for discussion.",
        "path": PROCESSED_DIR / "recovery_interpretation_cases.csv",
    },
    {
        "sheet_name": "10_dictionary",
        "description": "Recovery validation data dictionary.",
        "path": PROCESSED_DIR / "recovery_validation_data_dictionary.csv",
    },
    {
        "sheet_name": "11_method_notes",
        "description": "Methodological notes.",
        "path": DIAGNOSTICS_DIR / "recovery_validation_methodological_notes.csv",
    },
]

used_sheets = set()

with pd.ExcelWriter(RECOVERY_VALIDATION_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "10 Recovery Validation Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for POSet recovery validation diagnostics.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Recovery validation audit workbook created:")
print(RECOVERY_VALIDATION_AUDIT_XLSX.resolve())

Recovery validation audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\10_Recovery_Validation_Audit.xlsx


In [15]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("10 RECOVERY VALIDATION COMPLETE")
print("=" * 80)

print("\nProcessed folder:")
print(PROCESSED_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nFigures folder:")
print(FIGURES_DIR.resolve())

print("\nAudit workbook:")
print(RECOVERY_VALIDATION_AUDIT_XLSX.resolve())

print("\nMain processed outputs:")
main_outputs = [
    "profile_recovery_validation_summary.csv",
    "profile_recovery_group_summaries.csv",
    "working_main_profile_recovery_validation_summary.csv",
    "working_main_profile_recovery_group_summaries.csv",
    "epsilon_recovery_validation_summary.csv",
    "epsilon_margin_recovery_validation_summary.csv",
    "working_main_epsilon_recovery_validation_summary.csv",
    "working_main_epsilon_margin_recovery_validation_summary.csv",
    "recovery_validation_takeaway_table.csv",
    "recovery_interpretation_cases.csv",
]

for file_name in main_outputs:
    path = PROCESSED_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nCompact takeaway table:")
display(recovery_validation_takeaway_table)

print("\nImportant notes:")
print("1. Recovery was not used as an ordering variable.")
print("2. Validation is descriptive/associational, not causal.")
print("3. Multiple recovery variants were exported to handle Recovery=0 and Recovery=20 special cases.")
print("4. Mismatch cases are useful discussion points, not errors.")

print("\nNext notebook:")
print("11_Sensitivity_Analysis.ipynb")

10 RECOVERY VALIDATION COMPLETE

Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Recovery_Validation

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\10_Recovery_Validation

Figures folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Outputs\Figures\Recovery_Validation

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\10_Recovery_Validation_Audit.xlsx

Main processed outputs:
- FOUND: profile_recovery_validation_summary.csv
- FOUND: profile_recovery_group_summaries.csv
- FOUND: working_main_profile_recovery_validation_summary.csv
- FOUND: working_main_profile_recovery_group_summaries.csv
- FOUND: epsilon_recovery_validation_summary.

,method,set_name,shock_id,setting,validation_variant,country_count,frontier_or_pareto_count,frontier_minus_non_frontier_mean_recovery,frontier_minus_non_frontier_median_recovery,layer_vs_recovery_spearman,interpretation
0,profile_poset,baseline_6_variables,2007,5_levels,all_recovery_available,25,8,-0.8235,1.0000,0.0260,Negative mean/median difference means Pareto c...
1,profile_poset,baseline_6_variables,2007,5_levels,affected_recovered_only_exclude_0_and_20,23,8,-0.2667,1.0000,-0.0386,Negative mean/median difference means Pareto c...
2,profile_poset,baseline_6_variables,2019,5_levels,all_recovery_available,32,14,-0.1508,0.0000,0.2813,Negative mean/median difference means Pareto c...
3,profile_poset,baseline_6_variables,2019,5_levels,affected_recovered_only_exclude_0_and_20,27,12,-0.2167,0.0000,0.3644,Negative mean/median difference means Pareto c...
4,epsilon_margin,baseline_6_variables,2007,margin_0.00,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...
5,epsilon_margin,baseline_6_variables,2007,margin_0.00,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...
6,epsilon_margin,baseline_6_variables,2007,margin_0.05,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...
7,epsilon_margin,baseline_6_variables,2007,margin_0.05,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...
8,epsilon_margin,baseline_6_variables,2007,margin_0.10,all_recovery_available,25,13,-2.1282,-0.5000,0.2318,Negative mean/median difference means frontier...
9,epsilon_margin,baseline_6_variables,2007,margin_0.10,affected_recovered_only_exclude_0_and_20,23,13,-1.4615,-0.5000,0.2050,Negative mean/median difference means frontier...



Important notes:
1. Recovery was not used as an ordering variable.
2. Validation is descriptive/associational, not causal.
3. Multiple recovery variants were exported to handle Recovery=0 and Recovery=20 special cases.
4. Mismatch cases are useful discussion points, not errors.

Next notebook:
11_Sensitivity_Analysis.ipynb
